# DATASET

In [1]:
import pandas as pd
import numpy as np
import chess

In [2]:
raw_data = pd.read_csv('dataset/chessData.csv')
labels = raw_data["Evaluation"]
moves = raw_data["FEN"]

In [3]:
# def centipawns_to_win_probability(centipawns : str) -> float:
#     return 1 / (1 + np.exp(-int(centipawns.replace('#', '')) / 400))

def centipawns_to_win_probability(centipawns : str) -> float:
    if '#' in centipawns:
        return 1.0 if '+' in centipawns else 0.0
    return 1 / (1 + np.exp(-int(centipawns) / 400))

def FEN_to_input_vector(fen : str) -> np.ndarray:
    board = chess.Board(fen)
    turn = board.turn 
    
    input_vector = np.zeros(768, dtype=np.float32) # Moins de RAM !
        
    # Optimisation : piece_map() ne boucle que sur les cases occupées (max 32) au lieu de 64
    for square, piece in board.piece_map().items():
        display_square = square if turn == chess.WHITE else chess.square_mirror(square)
        
        piece_type = piece.piece_type - 1
        # Si c'est la pièce du joueur dont c'est le tour, offset = 0, sinon 6
        color_offset = 0 if piece.color == turn else 6               
                    
        input_vector[display_square * 12 + color_offset + piece_type] = 1   
            
    return input_vector

In [6]:
LABELS = labels.to_numpy()

labels_probabilities = np.vectorize(centipawns_to_win_probability)

LABELS = labels_probabilities(LABELS)



In [4]:
MOVES = moves.to_numpy()

MOVES = [FEN_to_input_vector(fen) for fen in MOVES]

In [17]:
MOVES[2]

array([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [10]:
mvs = np.array(MOVES, dtype=np.float32)

MemoryError: Unable to allocate 37.1 GiB for an array with shape (12958035, 768) and data type float32

In [16]:
np.savez_compressed('D:/Projects/NNUE chess/dataset.npz', X=MOVES, Y=LABELS)

MemoryError: Unable to allocate 37.1 GiB for an array with shape (12958035, 768) and data type float32

In [ ]:
mmapped_array = np.memmap('large_array.dat', dtype='float32', mode='w+', shape=(12958035, 768))

In [18]:
np.savez('labels.npz', Y=LABELS)

In [4]:
from scipy import sparse

def fens_to_sparse_matrix(fens):
    rows = []
    cols = []
    data = []
    for i, fen in enumerate(fens):
        board = chess.Board(fen)
        turn = board.turn
        
        for square, piece in board.piece_map().items():
            # Miroir si c'est aux noirs de jouer pour la symétrie
            display_square = square if turn == chess.WHITE else chess.square_mirror(square)
            piece_type = piece.piece_type - 1
            color_offset = 0 if piece.color == turn else 6
            
            index = display_square * 12 + color_offset + piece_type
            
            rows.append(i)
            cols.append(index)
            data.append(1.0)
            
    return sparse.csr_matrix((data, (rows, cols)), shape=(len(fens), 768), dtype=np.float32)

MOVES_sparse = fens_to_sparse_matrix(moves.to_numpy())

sparse.save_npz('moves.npz', MOVES_sparse)